# 03 — Evaluation harness

**Outcome:** partition and seal truth, freeze the event-level evaluation
policy, expose statistical resolution, and prove the matcher with
adversarial controls before any model is selected.

Point-adjusted accuracy is intentionally excluded. One incident receives
at most one fault credit.


## 1. Setup and policy


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT") or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_11_run1",
    "petrobras_3w": "petrobras_3w_core_v0_11_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
import evaluation_core
evaluation_core = importlib.reload(evaluation_core)
from evaluation_core import (
    ALERT_COLUMNS, EVALUATION_CORE_VERSION, evaluate_alerts,
    evaluate_cases, form_cases, partition_truth,
)

RUN_ROOT = (
    DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR
    / os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
)
CORE_ROOT, EVAL_ROOT, SPLIT_ROOT = (
    RUN_ROOT / "SPEC-CORE", RUN_ROOT / "SPEC-EVAL", RUN_ROOT / "SPLITS"
)
EDA_VERSION = EVALUATION_VERSION = "3.0.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / f"{SECTOR}_eda_v3_run1"
OUTPUT_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / f"{SECTOR}_evaluation_v3_run1"

BASE_POLICY = {
    "telecom": {
        "decision_horizon_seconds": 48 * 3600,
        "exposure_unit": "entity_day",
        "false_case_budget": 0.01,
        "calibration_block_seconds": 24 * 3600,
        "recovery_seconds": 1800,
        "case_gap_seconds": 3600,
        "threshold_quantiles": [0.95, 0.975, 0.99],
        "persistence": {
            "rapid_residual": 1800, "drift_cusum": 900,
            "peer_deviation": 1800, "group_common_mode": 1800,
            "dispersion_change": 1800,
        },
    },
    "petrobras_3w": {
        "decision_horizon_seconds": 6 * 3600,
        "decision_horizon_seconds_by_fault_type": {
            "Abrupt Increase of BSW": 12 * 3600,
            "Spurious Closure of DHSV": 20 * 60,
            "Severe Slugging": 5 * 3600,
            "Flow Instability": 15 * 60,
            "Rapid Productivity Loss": 12 * 3600,
            "Quick Restriction in PCK": 15 * 60,
            "Scaling in PCK": 72 * 3600,
            "Hydrate in Production Line": 5 * 3600,
            "Hydrate in Service Line": 5 * 3600,
        },
        "exposure_unit": "episode",
        "false_case_budget": 0.10,
        "calibration_block_seconds": None,
        "recovery_seconds": 300,
        "case_gap_seconds": 900,
        "threshold_quantiles": [0.90, 0.95, 0.975],
        "persistence": {
            "rapid_residual": 60, "drift_cusum": 10,
            "dispersion_change": 60,
        },
    },
}

if not EVAL_ROOT.is_dir():
    raise FileNotFoundError("SPEC-EVAL is required to build the harness")
manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.is_file() else pd.DataFrame()
selection_channels = ["rapid_residual", "drift_cusum"]
if not topology.empty:
    selection_channels += ["peer_deviation", "group_common_mode"]
selection_channels += ["dispersion_change"]
display(pd.Series({
    "sector": SECTOR, "truth": EVAL_ROOT,
    "topology_capability": not topology.empty,
    "candidate_channels": selection_channels,
}, name="value").to_frame())


## 2. Partition truth and inspect denominators


In [ ]:
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
episodes = pd.read_parquet(CORE_ROOT / "observation_episodes.parquet")
events = pd.read_parquet(EVAL_ROOT / "fault_events.parquet")
intervals = pd.read_parquet(EVAL_ROOT / "fault_entity_intervals.parquet")
condition_path = EVAL_ROOT / "condition_states.parquet"
conditions = pd.read_parquet(condition_path) if condition_path.is_file() else pd.DataFrame()

time_path = SPLIT_ROOT / "time_partitions.parquet"
entity_path = SPLIT_ROOT / "entity_partitions.parquet"
time_partitions = pd.read_parquet(time_path) if time_path.is_file() else pd.DataFrame()
entity_partitions = pd.read_parquet(entity_path) if entity_path.is_file() else pd.DataFrame()
primary_split = "time" if not time_partitions.empty else "entity"

truth, truth_audit, truth_summary = partition_truth(
    events, intervals, conditions, registry,
    primary_split=primary_split,
    time_partitions=time_partitions,
    entity_partitions=entity_partitions,
)
display(truth_summary)

development_events = truth["development"]["fault_events"]
development_faults = len(development_events)
shared_faults = development_events.fault_id.isin(
    truth["development"]["fault_entity_intervals"]
    .groupby("fault_id").entity_id.nunique().loc[lambda count: count.gt(1)].index
).sum()
one_fault_step = 1 / development_faults if development_faults else np.nan

if primary_split == "time":
    row = time_partitions.loc[time_partitions.partition.eq("development")].iloc[0]
    days = (pd.to_datetime(row.end_ts, utc=True) - pd.to_datetime(row.start_ts, utc=True)).total_seconds() / 86400
    exposure = days * registry.entity_id.nunique()
else:
    development_entities = set(entity_partitions.loc[
        entity_partitions.partition.eq("development"), "entity_id"
    ].astype(str))
    exposure = episodes.entity_id.astype(str).isin(development_entities).sum()

resolution = pd.DataFrame([{
    "development_faults": development_faults,
    "development_multi_entity_faults": int(shared_faults),
    "one_fault_recall_step": one_fault_step,
    "development_exposure": exposure,
    "exposure_unit": BASE_POLICY[SECTOR]["exposure_unit"],
    "one_false_case_rate_step": 1 / exposure if exposure else np.nan,
    "zero_false_cases_one_sided_95_upper": -np.log(0.05) / exposure if exposure else np.nan,
    "localisation_reporting": "estimable" if shared_faults >= 20 else "descriptive_only",
}])
display(resolution)


## 3. Adversarial controls


In [ ]:
BASE = pd.Timestamp("2025-01-01", tz="UTC")
event_columns = [
    "fault_id", "fault_type", "domain_type", "domain_id", "onset_ts",
    "observable_ts", "impact_ts", "end_ts", "group_id",
    "label_source", "source_instance_id",
]
test_events = pd.DataFrame([
    ("F1", "fault_a", "entity", "asset-1", BASE, BASE,
     BASE + pd.Timedelta(minutes=30), BASE + pd.Timedelta(hours=2), None, "test", "one"),
    ("F2", "fault_b", "entity", "asset-1", BASE + pd.Timedelta(minutes=10),
     BASE + pd.Timedelta(minutes=10), pd.NaT, BASE + pd.Timedelta(hours=2), None, "test", "two"),
], columns=event_columns)
test_intervals = pd.DataFrame([
    ("F1", "asset-1", BASE, BASE + pd.Timedelta(hours=2), "test", "one"),
    ("F2", "asset-1", BASE + pd.Timedelta(minutes=10), BASE + pd.Timedelta(hours=2), "test", "two"),
], columns=["fault_id", "entity_id", "start_ts", "end_ts", "label_source", "source_instance_id"])

def alert(alert_id, model, entity, minute, score=2.0, scope_type=None, scope_id=None, affected_fraction=None):
    start = BASE + pd.Timedelta(minutes=minute)
    return (alert_id, model, entity, "episode-1", start,
            start + pd.Timedelta(minutes=1), start, score, 1,
            "metric__level", scope_type, scope_id, affected_fraction)

alerts = pd.DataFrame([
    alert("A1", "rapid", "asset-1", 15),
    alert("A2", "drift", "asset-1", 20),
], columns=ALERT_COLUMNS)
result = evaluate_alerts(
    alerts, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
assert result["fault_results"].detected.sum() == 2

empty_alerts = pd.DataFrame(columns=ALERT_COLUMNS)
empty_result = evaluate_alerts(
    empty_alerts, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert empty_result["fault_results"].detected.sum() == 0
late_alerts = pd.DataFrame([
    alert("L1", "rapid", "asset-1", 180),
], columns=ALERT_COLUMNS)
late_result = evaluate_alerts(
    late_alerts, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert late_result["fault_results"].detected.sum() == 0

cases, members = form_cases(
    alerts, gap_seconds=600, thresholds={"rapid": 1.0, "drift": 1.0}
)
assert len(cases) == 1
case_result = evaluate_cases(
    cases, members, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
assert case_result["fault_results"].detected.sum() == 1

topology_fixture = pd.DataFrame([
    ("a", "segment", "AB", 0, "physical_topology"),
    ("b", "segment", "AB", 0, "physical_topology"),
    ("b", "segment", "BC", 0, "physical_topology"),
    ("c", "segment", "BC", 0, "physical_topology"),
], columns=["entity_id", "group_type", "group_id", "hierarchy_level", "group_family"])
chain_alerts = pd.DataFrame([
    alert("B1", "group_common_mode", "a", 0, scope_type="segment", scope_id="AB"),
    alert("B2", "group_common_mode", "b", 1, scope_type="segment", scope_id="AB"),
    alert("B3", "group_common_mode", "c", 2, scope_type="segment", scope_id="BC"),
], columns=ALERT_COLUMNS)
chain_cases, _ = form_cases(
    chain_alerts, topology_fixture, gap_seconds=600,
    thresholds={"group_common_mode": 1.0}
)
assert len(chain_cases) == 2

run_manifest = read_json(RUN_ROOT / "run_manifest.json")
assert run_manifest["evaluation_mounted"]
assert not any((CORE_ROOT / f"{name}.parquet").exists() for name in [
    "fault_events", "fault_entity_intervals", "condition_states"
])
display(pd.DataFrame({
    "test": ["overlapping matching", "empty-score control",
             "late-alert control", "one case-one credit",
             "no transitive topology chain", "SPEC-CORE truth boundary"],
    "status": "pass",
}))


## 4. Freeze policy and physically seal holdout


In [ ]:
policy = {
    "evaluation_version": EVALUATION_VERSION,
    "evaluation_core_version": EVALUATION_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "primary_split": primary_split,
    **BASE_POLICY[SECTOR],
    "selection_channels": selection_channels,
    "channel_persistence_seconds": BASE_POLICY[SECTOR]["persistence"],
    "cusum_allowance": 0.5,
    "budget_safety_factor": 0.90,
    "equivalence_margin": 0.02,
    "threshold_method": "empirical quantile of calibration block maxima",
    "matching": "same affected entity and deterministic one-to-one event credit",
    "localisation": "exact, top-two, equivalence-aware, hierarchy and footprint metrics",
    "synthetic_limit": (
        "Telecom results validate injected mechanisms, not real-fleet effectiveness"
        if SECTOR == "telecom" else
        "Petrobras qualifies the topology-free common path"
    ),
    "development_fault_count": development_faults,
    "development_multi_entity_faults": int(shared_faults),
    "one_fault_recall_step": one_fault_step,
    "development_exposure": exposure,
    "holdout_used": False,
}

with new_output_directory(OUTPUT_ROOT) as output:
    for partition, folder in (("development", "development"), ("holdout", "holdout_sealed")):
        target = output / folder
        target.mkdir()
        for table_name, frame in truth[partition].items():
            if not frame.empty:
                frame.to_parquet(target / f"{table_name}.parquet", index=False)
    truth_audit.to_csv(output / "truth_partition_audit.csv", index=False)
    resolution.to_csv(output / "statistical_resolution.csv", index=False)
    write_json(output / "evaluation_policy.json", policy)

display(pd.Series(policy, name="value").to_frame())
print("PASS — evaluation policy frozen before modelling")
print("Saved:", OUTPUT_ROOT)
print("Next: 04_SIMPLE_ANOMALY_MODELS.ipynb")
